# Hardware validation: more and deeper kernel-fidelity evidence
Three short IBM Quantum runs, about **4–5 QPU-minutes in total** (within the free plan's 10 minutes per month):

1. **z12**: about 400 kernel entries with the paper's Z-map circuit at 12 qubits, from 3 seeds.
2. **zz12**: about 300 entries with the *entangling* ZZ1 circuit at 12 qubits (depth about 200, 42 two-qubit gates).
3. **repeat**: about 120 Z-map entries on a *different* IBM device, to show stability across devices and days.

A GPU is not needed: the default CPU runtime is fine. Results are saved to Google Drive (folder `hw_validation`); if Colab disconnects, run the cells again from the top and finished runs are skipped. Most of the waiting is IBM's queue, not computing. Keep the tab open.

**If Colab disconnects after a job was submitted:** just run the cells again from the top. Each job's ID is saved in Drive when it is submitted, so the same job is collected and no second job is sent. You can watch your jobs at quantum.cloud.ibm.com → Workloads.

In [ ]:
from google.colab import drive, files
import os, shutil
drive.mount('/content/drive')
D = '/content/drive/MyDrive/hw_validation'; os.makedirs(D, exist_ok=True)
if not os.path.exists(f'{D}/hw_bundle_v2.zip'):
    up = files.upload()  # choose hw_bundle_v2.zip (first time only)
    shutil.copy('hw_bundle_v2.zip', f'{D}/hw_bundle_v2.zip')
!unzip -oq {D}/hw_bundle_v2.zip -d /content/run && ls /content/run

In [ ]:
!pip -q install --ignore-installed PyJWT
!pip -q install 'qiskit==2.*' qiskit-ibm-runtime qiskit-aer

**Credentials.** Paste your IBM Quantum API key when asked: it is hidden and never saved. Then paste your instance CRN (from quantum.cloud.ibm.com → Instances).

In [ ]:
from getpass import getpass
TOKEN = getpass('IBM Quantum API key: ')
INSTANCE = input('Instance CRN: ').strip()

## Run 1: Z map, 12 qubits (about 75 QPU-seconds)

In [ ]:
JOB1 = 'dbpk6r0qtr0c738eip8g'   # job already submitted before the disconnect; set to '' to submit a new one
extra = f'--job_id {JOB1}' if JOB1 else ''
%cd /content/run
!python -u hw_validation.py --tag z12 --map Z --seeds 0 1 2 --pairs 135 --out {D}/results --token {TOKEN} --instance {INSTANCE} {extra} 2>&1 | grep -v -i warning

## Run 2: entangling ZZ1 map, 12 qubits (about 1–2 QPU-minutes)

In [ ]:
%cd /content/run
!python -u hw_validation.py --tag zz12 --map ZZ1 --seeds 0 1 2 --pairs 100 --out {D}/results --token {TOKEN} --instance {INSTANCE} 2>&1 | grep -v -i warning

## Run 3: repeat on a different device (about 25 QPU-seconds)
It automatically avoids the device used in Run 1. Running this cell on a later day as well is even better.

In [ ]:
import json
first = json.load(open(f'{D}/results/z12.json'))['backend']; print('avoiding', first)
%cd /content/run
!python -u hw_validation.py --tag repeat --map Z --seeds 0 --pairs 120 --avoid {first} --out {D}/results --token {TOKEN} --instance {INSTANCE} 2>&1 | grep -v -i warning

## Download
Attach `hw_validation_results.zip` in the chat. Copies stay in Drive → `hw_validation/results`.

In [ ]:
!cd {D} && zip -qr /content/hw_validation_results.zip results
files.download('/content/hw_validation_results.zip')